In [ ]:
import pandas as pd

from egomimic.scripts.plotting.plotting import (
    ColorsPalette,
    plot_bar_chart,
    plot_multi_line_chart,
    plot_specified_bars,
)

## Line Plots

In [ ]:
# Dummy values example
lines = {
    "Series A": ([0, 1, 2, 3, 4], [1, 2, 1, 3, 5], ColorsPalette.PACBLUE[4], [0.2, 0.3, 0.2, 0.4, 0.3]),
    "Series B": ([0, 1, 2, 3, 4], [2, 1, 2, 2, 4], ColorsPalette.WILLOWGREEN[4], [0.3, 0.2, 0.3, 0.2, 0.4]),
    "Series C": ([0, 1, 2, 3, 4], [0, 1, 0, 1, 0], ColorsPalette.TIGERFLAME[4], [0.1, 0.2, 0.1, 0.2, 0.1]),
    "Series D": ([0, 1, 2, 3, 4], [0, 1, 3, 1, 2], ColorsPalette.LILAC[4], [0.2, 0.5, 0.2, 0.2, 0.1]),
}
plot_multi_line_chart(lines, "Time", "Value", "Dummy Multi-Line Chart")

## Bar Charts

In [ ]:
df_flagship = pd.read_csv("flagship.csv", skiprows=2)
df_flagship


In [ ]:
from egomimic.scripts.plotting.plotting import print_flagship_latex_table

latex = print_flagship_latex_table(
    df_flagship,
    teams=["Robot A", "Robot B", "Robot C"],
    tasks=["Object_in_container", "Put_cup_on_saucer", "Bag_groceries"],
    caption="Flagship benchmark results.",
    label="tab:flagship",
)

print(latex)

In [ ]:
team_map = {"GT": "Robot A", "Stanford": "Robot B", "UCSD": "Robot C"}
df_flagship["Team"] = df_flagship["Team"].map(team_map).fillna(df_flagship["Team"])
df_flagship = df_flagship[df_flagship["Team"] != "ETH"].copy()
df_flagship["Robot-Only (OOD)"] = (df_flagship["Robot-Only (New Object)"] + df_flagship["Robot-Only (New Object + Scene)"]) / 2.0
df_flagship["Co-train (OOD)"] = (df_flagship["Co-train (New Object)"] + df_flagship["Co-train (New Object + Scene)"]) / 2.0


In [ ]:
# Build grouped bar chart from flagship table
def _to_float(value):
    if pd.isna(value):
        return 0.0
    if isinstance(value, str):
        cleaned = value.strip()
        if cleaned == "":
            return 0.0
        if cleaned.endswith("%"):
            try:
                return float(cleaned[:-1]) / 100.0
            except ValueError:
                return 0.0
        try:
            return float(cleaned)
        except ValueError:
            return 0.0
    try:
        return float(value)
    except (TypeError, ValueError):
        return 0.0
def _prepare_in_domain_data(df, tasks, in_domain_robot_col, in_domain_cotrain_col):
    df_use = df[df["Task Name"].isin(tasks)].copy()
    if df_use.empty:
        raise ValueError("No rows matched the provided tasks")
    df_use["Task Name"] = pd.Categorical(df_use["Task Name"], categories=tasks, ordered=True)
    norm = df_use["Normalization"].apply(_to_float).replace(0, 1.0)
    for col in (in_domain_robot_col, in_domain_cotrain_col):
        df_use[col] = df_use[col].apply(_to_float)
        df_use[col] = (df_use[col] / norm).fillna(0.0)
    return df_use
# def _select_teams(df_use, max_teams=3, exclude=("ETH",)):
#     teams = [t for t in df_use["Team"].dropna().unique().tolist() if t not in exclude]
#     if not teams:
#         teams = df_use["Team"].dropna().unique().tolist()
#     if max_teams is not None and len(teams) > max_teams:
#         teams = teams[:max_teams]
#     return teams
def _team_colors(num_teams):
    base = [ColorsPalette.PACBLUE[4], ColorsPalette.TIGERFLAME[4], ColorsPalette.LILAC[4]]
    return base[:num_teams]
def _build_groups(df_use, tasks, teams, in_domain_robot_col, in_domain_cotrain_col, bar_colors):
    groups = {}
    for task in tasks:
        task_rows = df_use[df_use["Task Name"] == task]
        robot_vals = []
        cotrain_vals = []
        for team in teams:
            row = task_rows[task_rows["Team"] == team]
            if row.empty:
                robot_vals.append(0.0)
                cotrain_vals.append(0.0)
            else:
                robot_vals.append(float(row[in_domain_robot_col].iloc[0]))
                cotrain_vals.append(float(row[in_domain_cotrain_col].iloc[0]))
        groups[f"{task} (Robot-only)"] = (robot_vals, None, bar_colors)
        groups[f"{task} (Co-train)"] = (cotrain_vals, None, bar_colors)
    return groups
def _pair_positions(num_pairs, within_pair=0.12, between_pairs=0.12):
    positions = []
    x = 0.0
    for _ in range(num_pairs):
        positions.extend([x, x + within_pair])
        x += within_pair + between_pairs
    max_x = positions[-1] if positions[-1] != 0 else 1.0
    return [p / max_x for p in positions]
def _apply_pair_labels(ax, group_x_spacing, tasks, group_labels, label_offset=-0.12, bottom_pad=0.22):
    ax.set_xticks(group_x_spacing)
    ax.set_xticklabels(group_labels * len(tasks), rotation=0, ha="center")
    pair_centers = [
        (group_x_spacing[i] + group_x_spacing[i + 1]) / 2
        for i in range(0, len(group_x_spacing), 2)
    ]
    for task, center in zip(tasks, pair_centers):
        ax.text(
            center,
            label_offset,
            task,
            transform=ax.get_xaxis_transform(),
            ha="center",
            va="top",
        )
    ax.figure.subplots_adjust(bottom=bottom_pad)
def plot_in_domain_pairs(df_orig, tasks, in_domain_robot_col, in_domain_cotrain_col, title, teams, *, within_pair=0.12, between_pairs=0.12):
    df_copy = df_orig.copy()
    df_use = _prepare_in_domain_data(df_copy, tasks, in_domain_robot_col, in_domain_cotrain_col)
    # teams = _select_teams(df_use)
    bar_colors = _team_colors(len(teams))
    groups = _build_groups(df_use, tasks, teams, in_domain_robot_col, in_domain_cotrain_col, bar_colors)
    group_x_spacing = _pair_positions(len(tasks), within_pair=within_pair, between_pairs=between_pairs)
    fig, ax = plot_bar_chart(
        groups,
        title=title,
        group_x_spacing=group_x_spacing,
        bar_labels=teams,
        legend_title="Teams",
        figsize=(10, 5),
        group_mean=True
    )
    _apply_pair_labels(ax, group_x_spacing, tasks, ["Robot-only", "Co-train"])
    return fig, ax


In [ ]:
tasks = ["Object_in_container", "Put_cup_on_saucer", "Bag_groceries"]
teams = ["Robot A", "Robot B", "Robot C"]
fig, ax = plot_in_domain_pairs(df_flagship, tasks, in_domain_robot_col="Robot-Only (ID)", in_domain_cotrain_col="Co-train (ID)", title="In-Domain Task Performance", teams=teams, within_pair=0.15, between_pairs=0.2)
fig.savefig("plot_out/in_domain_task_performance.png", dpi=300)


In [ ]:
tasks = ["Object_in_container", "Put_cup_on_saucer", "Bag_groceries"]
fig, ax = plot_in_domain_pairs(df_flagship, tasks, in_domain_robot_col="Robot-Only (OOD)", in_domain_cotrain_col="Co-train (OOD)", title="OOD Task Performance", teams=teams, within_pair=0.15, between_pairs=0.2)
fig.savefig("plot_out/ood_task_performance.png", dpi=300)


In [ ]:
df_scaling = pd.read_csv("scaling_law.csv")
columns = [c for c in df_scaling.columns if c != "Normalization"]
df_scaling[columns] = df_scaling[columns].fillna(0)
rename_map = {
    "0h (robot only)": "Robot only",
    "1h in-domain": "ID (1hr)",
    "2h in-domain": "ID (2hr)",
    "4h (2h in-domain + 2h egoverse 1 lab)": "EV (2hr) + ID (2hr)",
    "8h (NO in-domain + 8h egoverse)": "EV (8hr)",
    "8h (2h in-domain + 8h egoverse)": "EV (8hr) + ID (2hr)",
}
df_scaling["Human Data (hrs)"] = df_scaling["Human Data (hrs)"].replace(rename_map)
task = "Object in Container"
preferred_order = [
    "EV (8hr) + ID (2hr)",
    "EV (8hr)",
    "EV (2hr) + ID (2hr)",
    "ID (2hr)",
    "ID (1hr)",
    "Robot only",
]
score_cols = ["In-domain (20 rollouts)", "New object (10 rollouts)", "New object + new scene (10 rollouts)"]


In [ ]:
df_scaling


In [ ]:
fig, ax = plot_specified_bars(df_scaling, task, preferred_order, score_cols=score_cols, bar_colors=[ColorsPalette.TIGERFLAME[7], ColorsPalette.TIGERFLAME[6], ColorsPalette.TIGERFLAME[5], ColorsPalette.TIGERFLAME[5], ColorsPalette.TIGERFLAME[3], ColorsPalette.TIGERFLAME[2]], legend_title=None, figsize=(8,5), group_width_scale=0.7)
fig.savefig("plot_out/scaling_law_object_in_container.png", dpi=300)


In [ ]:
fig, ax = plot_specified_bars(df_scaling, "Fold Clothes", preferred_order, score_cols=score_cols, bar_colors=[ColorsPalette.PACBLUE[7], ColorsPalette.PACBLUE[6], ColorsPalette.PACBLUE[5], ColorsPalette.PACBLUE[5], ColorsPalette.PACBLUE[3], ColorsPalette.PACBLUE[2]], legend_title=None, figsize=(8,5), group_width_scale=0.7)
fig.savefig("plot_out/scaling_law_fold_clothes.png", dpi=300)


In [ ]:
fig, ax = plot_specified_bars(df_scaling, "Bag Groceries", preferred_order, score_cols=score_cols, bar_colors=[ColorsPalette.LILAC[7], ColorsPalette.LILAC[6], ColorsPalette.LILAC[5], ColorsPalette.LILAC[5], ColorsPalette.LILAC[3], ColorsPalette.LILAC[2]], legend_title=None, figsize=(8,5), group_width_scale=0.7)
fig.savefig("plot_out/scaling_law_bag_groceries.png", dpi=300)
